# Growth across Europe's regions: convergence between countries, divergence within them

**Real data, 223 regions in 25 countries.** Notebook `11` ran the standard
sequence on Portuguese municipalities, where every unit shares one national
economy. This one runs it on the NUTS-2 regions of the EU, where they do not.
That difference is the subject of the notebook.

The question is the oldest one in regional economics: **did poorer regions
grow faster between 2012 and 2022?** That is β-convergence. Regress growth on
the starting level of income, and a negative slope means the poor are catching
up. Regional growth is spatially clustered, so the regression needs a spatial
model, and the package can fit one in a line.

What the notebook shows:

1. **OLS finds convergence and the spatial lag removes it.** The slope on
   initial income is −0.39 without the lag and −0.04 with it.
2. **The spatial dependence is mostly national.** Country dummies explain 60%
   of the variance in growth. Once they are in, Moran's I on the residuals goes
   *negative*, and the convergence slope flips to **+1.01**: within a country,
   the richer regions pulled ahead.
3. **The old and new member states are different economies.** A regime split
   declared in advance rejects, and the divergence is concentrated in the
   countries that joined in 2004 or later.
4. **One country's national accounts can move ρ̂ by a tenth.** Ireland's three
   regions top the growth table, and that is profit accounting by
   multinationals, not local production.

Data and licensing are in `data/europe_PROVENANCE.md`. The Eurostat and
OpenStreetMap credits are conditions of their licences. Keep them with
anything you publish from this.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

using SpatialEconometrics
using Printf, Statistics, LinearAlgebra, SparseArrays, Random
using StatsAPI: vcov

const DATA = joinpath(@__DIR__, "data")

## 1. Load the data and check the join

The panel holds one row per region and year. The region file holds names,
centroids and land areas. The GAL file holds queen contiguity. All three are
sorted by NUTS code, and `read_gal` returns the ids it read, so the check is
one comparison.

In [ ]:
function readcsv(path)
    lines = readlines(path)
    hdr = split(lines[1], ',')
    rows = [split(l, ','; limit = length(hdr)) for l in lines[2:end] if !isempty(l)]
    return hdr, rows
end

ph, prows = readcsv(joinpath(DATA, "europe_nuts2.csv"))
rh, rrows = readcsv(joinpath(DATA, "europe_nuts2_regions.csv"))

panel = Dict((r[1], parse(Int, r[3])) => r for r in prows)
function val(id, year, column)
    r = get(panel, (id, year), nothing)
    j = findfirst(==(column), ph)
    (r === nothing || isempty(r[j])) ? NaN : parse(Float64, r[j])
end

ids     = [r[1] for r in rrows]
country = [r[2] for r in rrows]
place   = [strip(r[end], '"') for r in rrows]
# EPSG:3035 centroids, converted from metres to kilometres
xy = hcat(parse.(Float64, [r[3] for r in rrows]), parse.(Float64, [r[4] for r in rrows])) ./ 1000

Wq, wid = read_gal(joinpath(DATA, "europe_nuts2_queen.gal"))
@printf("regions %d in %d countries; panel rows %d, years %d-%d\n",
        length(ids), length(unique(country)), length(prows),
        minimum(parse(Int, r[3]) for r in prows), maximum(parse(Int, r[3]) for r in prows))
@printf("GAL ids match the region file in order: %s\n", wid == ids)
@printf("regions with no land neighbour: %s\n", join(ids[vec(sum(Wq; dims = 2)) .== 0], " "))

The eleven regions with no land neighbour are islands: Cyprus, Malta, Corsica,
Sardinia, Sicily, the Balearics, Åland, Crete, the Ionian and North and South
Aegean islands. A contiguity matrix gives them all-zero rows. They are dropped
from the contiguity models below and come back in section 7, where k-nearest
neighbours gives every region neighbours.

## 2. The variables, and the regions they cost

Three variables, all from Eurostat's regional accounts:

- **growth**: mean annual growth of real GDP, 2012–2022, in percent, from the
  chain-linked volume index;
- **log GDP per head in PPS in 2012**: the starting level of income, at
  purchasing-power parity, so that a euro in Bucharest and a euro in Munich
  are put on the same footing;
- **investment share**: gross fixed capital formation as a percentage of GDP,
  averaged over 2012–2021.

The window starts in 2012 because Romania's regional GDP per head in PPS
starts in 2012. A 2010 start would lose the eight Romanian regions, and they
are the fastest-converging ones in the sample.

In [ ]:
const T0, T1 = 2012, 2022

growth = [100 * log(val(id, T1, "real_gdp_index") / val(id, T0, "real_gdp_index")) / (T1 - T0)
          for id in ids]
y0     = [log(val(id, T0, "gdp_pps_per_head")) for id in ids]
invest = [100 * mean(filter(isfinite, [val(id, t, "gfcf_mio_eur") / val(id, t, "gdp_mio_eur")
                                       for t in T0:T1-1])) for id in ids]

complete = findall(i -> isfinite(growth[i]) && isfinite(y0[i]) && isfinite(invest[i]),
                   eachindex(ids))
println("no usable data: ", join(setdiff(ids, ids[complete]), " "))

# Contiguity sample: complete regions with at least one complete neighbour.
# Subset the binary graph first and row-standardise last.
B      = Wq[complete, complete] .> 0
linked = complete[vec(sum(B; dims = 2)) .> 0]
W      = row_standardize(sparse(Float64.(Wq[linked, linked] .> 0)))
@assert all(>(0), sum(W; dims = 2))
println("complete but without a neighbour: ", join(setdiff(ids[complete], ids[linked]), " "))

y  = growth[linked]
X  = hcat(y0[linked], invest[linked])
vn = ["log_gdp_pps_2012", "investment_share"]
cc = country[linked]
N  = length(linked)
@printf("\ncomplete %d, in the contiguity sample %d\n", length(complete), N)

Switzerland has no regional volume index or investment series, and six of
Norway's seven regions have no volume index. They drop out for want of data,
not by choice. The eleven island regions drop out of the contiguity sample
only. That leaves **234 regions with data and 223 with a land neighbour**.

In [ ]:
@printf("real growth, %%/year: mean %.2f, median %.2f, sd %.2f\n", mean(y), median(y), std(y))
ord = sortperm(y; rev = true)
println("\nfastest:")
for i in ord[1:6]
    @printf("  %-5s %-34s %5.2f\n", ids[linked[i]], first(place[linked[i]], 34), y[i])
end
println("slowest:")
for i in ord[end-5:end]
    @printf("  %-5s %-34s %5.2f\n", ids[linked[i]], first(place[linked[i]], 34), y[i])
end

Look at the top of the table before you fit anything. Three of the six fastest
"regions" are Irish, with Southern Ireland at **11% a year**. That is not
production in Cork. It is the profits of multinationals that book their
intellectual property in Ireland and show up in its GDP. Section 4 measures
what those regions do to the estimates.

The bottom of the table is easier to explain. Dytiki Makedonia, at −5.4% a
year, is Greece's lignite region, where mines and power stations closed.
Groningen, at −1.2%, is the Dutch gas field, run down after the earthquakes.
Both are single-industry regions that lost the industry.

## 3. Spatial dependence, and the first model

In [ ]:
println("growth          ", moran_global(y, W))
println("log GDP/head    ", moran_global(X[:, 1], W))
println("investment      ", moran_global(X[:, 2], W))

All three variables are strongly clustered. Growth has I = 0.50 (z = 10.7),
initial income 0.62 and the investment share 0.32. Rich regions sit next to
rich regions, and fast-growing regions next to fast-growing regions.

In [ ]:
Xc    = hcat(ones(N), X)
b_ols = Xc \ y
e_ols = y - Xc * b_ols
@printf("OLS  intercept %7.3f   log GDP/head %7.3f   investment %6.4f   R2 %.3f\n",
        b_ols..., 1 - sum(abs2, e_ols) / sum(abs2, y .- mean(y)))
println("\nMoran's I on the OLS residuals:")
println(moran_global(e_ols, W))

**OLS finds convergence.** The slope on log initial income is −0.386: a
region that started 10% richer grew 0.04 percentage points a year slower.
R² is 0.095, so most of the variation in growth is something else.

And most of the clustering is still there. Moran's I on the residuals is
**0.44** against 0.50 on raw growth. Initial income and investment explain
little of the spatial pattern.

In [ ]:
lms = lm_tests(y, Xc, W)
for k in (:lag, :error, :robust_lag, :robust_error)
    println(getfield(lms, k))
end

Both classical statistics reject, as they usually do. The robust pair
discriminates: **robust LM-lag is 15.1 (p = 0.0001), robust LM-error is 2.5
(p = 0.12)**. The battery points at a spatial lag. Note the prediction, and
see whether the likelihood agrees.

In [ ]:
m_sar = fit_spatial(SAR(), W, X, y; varnames = vn, method = ExactLU())
m_sem = fit_spatial_error(W, X, y;  varnames = vn, method = ExactLU())
m_sdm = fit_spatial(SDM(), W, X, y; varnames = vn, method = ExactLU())
println(regression_table(m_sar, m_sem; model_names = ["SAR", "SEM"]))
println(model_comparison_table(m_sar, m_sdm, m_sem; names = ["SAR", "SDM", "SEM"]))
println(vuong_test(m_sar, m_sem))

**The convergence slope disappears in SAR**: −0.043, with a standard error of
0.193. Under SEM it is +0.64 and significant at 5%, so the two models do not
even agree on the sign. The investment share is the one stable result: 0.080
under SAR and 0.079 under SEM, both with p < 0.001.

The likelihood does not settle the specification. SEM beats SAR by 3.1 AIC
points, against the robust LM prediction, and the Vuong test puts the two at
−0.39 (p = 0.70): indistinguishable. SDM beats both, with AIC 705.0 and the
lowest BIC too. On Portuguese municipalities the battery and the likelihood
agreed. Here they do not, and the honest report is that the data do not choose
between a lag and an error process.

What *is* settled is what OLS was doing. A region's growth tracks its
neighbours' growth (ρ̂ = 0.575), and poor regions sit next to poor regions.
OLS attributed to a region's own income what belongs to its neighbourhood.
Give the neighbourhood its own term and initial income has nothing left to
explain.

In [ ]:
println(spatial_effects(m_sar))

In a lag model the coefficient is not the effect. For the investment share,
the direct effect is 0.089 and the **indirect effect is 0.099**, for a total of
0.188: more than twice the coefficient of 0.080. Investment is a regressor
that nobody would call exogenous, because fast growth raises investment as
much as investment raises growth. Read the number as an association that
spreads across borders, not as the return on a euro of investment.

### Ireland

Ireland's three regions are the outliers identified in section 2. Leave them
out and refit.

In [ ]:
notIE = findall(!=("IE"), cc)
Bie   = Wq[linked[notIE], linked[notIE]] .> 0
@assert all(>(0), sum(Bie; dims = 2))
Wie   = row_standardize(sparse(Float64.(Bie)))
b_ie  = hcat(ones(length(notIE)), X[notIE, :]) \ y[notIE]
m_ie  = fit_spatial(SAR(), Wie, X[notIE, :], y[notIE]; varnames = vn, method = ExactLU())
se_ie = sqrt.(diag(vcov(m_ie)))
@printf("%-22s %5s %14s %16s %8s\n", "", "N", "OLS log GDP", "SAR log GDP", "SAR rho")
@printf("%-22s %5d %14.3f %8.3f (%.3f) %8.3f\n", "all regions", N, b_ols[2],
        m_sar.coef[2], sqrt(vcov(m_sar)[2, 2]), m_sar.rho)
@printf("%-22s %5d %14.3f %8.3f (%.3f) %8.3f\n", "Ireland left out", length(notIE), b_ie[2],
        m_ie.coef[2], se_ie[2], m_ie.rho)

The convergence slope is still insignificant under SAR (−0.139, standard error
0.184). **ρ̂ falls from 0.575 to 0.476.** Three adjacent regions with extreme
growth contribute that much spatial dependence, because each one is the
other two's neighbourhood. If a spillover estimate depends on one country,
report it with and without that country.

## 4. How much of the spatial dependence is national?

Regions in the same country share a currency (or a monetary policy), a tax
system, a fiscal response to the euro crisis and a set of national accounts.
Regions also sit next to other regions in their own country. A spatial lag
over contiguity cannot tell "my neighbour grew" from "my country grew".
Country dummies can.

In [ ]:
countries = sort(unique(cc))
Dc  = hcat([Float64.(cc .== c) for c in countries[2:end]]...)
Xf  = hcat(Xc, Dc)
b_f = Xf \ y
e_f = y - Xf * b_f
@printf("%d countries, %d dummies\n", length(countries), size(Dc, 2))
@printf("OLS with country dummies: log GDP/head %7.3f   investment %6.4f\n", b_f[2], b_f[3])
@printf("share of the variance in growth between countries: %.2f\n",
        1 - sum(abs2, y - Dc * (Dc \ (y .- mean(y))) .- mean(y)) / sum(abs2, y .- mean(y)))
println("\nMoran's I on the residuals:")
println(moran_global(e_f, W))
lmf = lm_tests(y, Xf, W)
for k in (:lag, :error, :robust_lag, :robust_error)
    println(getfield(lmf, k))
end

**Country dummies alone explain 60% of the variance in regional growth.** The
other 40% is the variation within countries.

With the dummies in, two things change.

**The convergence slope flips sign, to +1.011.** Across Europe, poorer
*countries* grew faster. Within a country, the *richer* regions grew faster.
The pooled −0.386 was an average of catch-up between countries and pull-ahead
within them. Neither one is "convergence".

**The residual dependence turns negative.** Moran's I is −0.137 (z = −2.8),
and the robust LM pair now points at the error term (5.9, p = 0.015, against
2.2, p = 0.14). Before the dummies, positive dependence was there. After them,
a region's growth is *negatively* related to its neighbours'.

In [ ]:
m_sarf = fit_spatial(SAR(), W, hcat(X, Dc), y; method = ExactLU())
sef    = sqrt.(diag(vcov(m_sarf)))
@printf("SAR with country dummies: rho %.3f (se %.3f)\n", m_sarf.rho, sef[end])
@printf("  log GDP/head %.3f (se %.3f)   investment %.4f (se %.4f)\n",
        m_sarf.coef[2], sef[2], m_sarf.coef[3], sef[3])

SAR agrees: **ρ̂ = −0.225** (standard error 0.086). Within a country, a region
grew faster when its neighbours grew slower. A capital region that draws
investment and people from its hinterland makes that pattern. The older
literature calls it a backwash effect. It is not the only explanation, but it
is the one the sign rules in.

The ρ̂ of 0.575 from section 3 is not a spillover between regions. It is
mostly the fact that regions in the same country grew at similar rates.

## 5. Old and new member states

The divergence within countries suggests a split. The 13 countries that joined
the EU in 2004 or later started poorer, grew faster and have dominant capital
regions. The split is declared here before the fit: Bulgaria, Croatia, Cyprus,
Czechia, Estonia, Hungary, Latvia, Lithuania, Malta, Poland, Romania, Slovakia
and Slovenia. Everything else in the contiguity sample is in the EU-15:
Norway and Switzerland have no regions with complete data.

In [ ]:
const NEW_MEMBERS = Set(["BG", "CY", "CZ", "EE", "HR", "HU", "LT", "LV", "MT",
                         "PL", "RO", "SI", "SK"])
labels = [c in NEW_MEMBERS ? 2 : 1 for c in cc]
@printf("joined 2004 or later: %d regions; EU-15: %d\n",
        count(==(2), labels), count(==(1), labels))
m_reg = fit_spatial_regime(W, X, y, ExogenousRegimes(labels); varnames = vn, method = ExactLU())
println(m_reg)
println(regime_chow_test(m_sar, m_reg))

**It rejects.** LR = 32.5 on 3 degrees of freedom.

| | EU-15 | joined 2004+ |
|---|---|---|
| log GDP per head 2012 | 0.47 (p = 0.11) | **1.13** (p = 0.002) |
| investment share | 0.059 (p = 0.013) | 0.072 (p = 0.031) |

In the new member states, a region 10% richer in 2012 grew 0.11 percentage
points a year *faster*. The capital regions are the pattern: Bucureşti-Ilfov,
Pest (around Budapest), Warsaw, Prague and Sofia's region, Yugozapaden, were
each the fastest-growing region in their country, pulling away from it while
the country caught up with the EU-15. In the EU-15 the slope is positive but
not significant.

ρ̂ falls from 0.575 to 0.481 once the regimes are in. Part of what the
spatial lag fitted was the difference between the two groups.

## 6. Two choices that look consequential

### Contiguity or nearest neighbours

The contiguity sample drops eleven islands. Six nearest neighbours, measured
between the EPSG:3035 centroids in kilometres, gives every region neighbours,
so all 234 regions stay in.

In [ ]:
W6 = knn_weights(xy[complete, :], 6)
m6 = fit_spatial(SAR(), W6, hcat(y0[complete], invest[complete]), growth[complete];
                 varnames = vn, method = ExactLU())
se6 = sqrt.(diag(vcov(m6)))
@printf("%-26s %5s %8s %14s %14s\n", "", "N", "rho", "log GDP/head", "investment")
@printf("%-26s %5d %8.3f %8.3f (%.3f) %7.4f (%.4f)\n", "queen, islands dropped", N,
        m_sar.rho, m_sar.coef[2], sqrt(vcov(m_sar)[2, 2]), m_sar.coef[3], sqrt(vcov(m_sar)[3, 3]))
@printf("%-26s %5d %8.3f %8.3f (%.3f) %7.4f (%.4f)\n", "6 nearest, islands kept", length(complete),
        m6.rho, m6.coef[2], se6[2], m6.coef[3], se6[3])

ρ̂ moves from 0.575 to 0.558, and the investment coefficient from 0.080 to
0.083. The convergence slope changes sign (−0.043 to +0.088) but is
insignificant both times. The islands do not drive the results.

### How wide is a Conley neighbourhood

The centroids are in kilometres on an equal-area projection, so the bandwidth
is a distance that means the same thing in Lapland and in Andalusia. Sweep it.

In [ ]:
se_base = sqrt.(diag(vcov(m_sar)))
@printf("%-10s %14s %14s %10s\n", "bw (km)", "se(log GDP)", "se(invest)", "se(rho)")
@printf("%-10s %14.4f %14.4f %10.4f   <- model-based\n", "-", se_base[2], se_base[3], se_base[4])
for bw in (100, 200, 300, 500, 800)
    se = sqrt.(diag(robust_vcov(m_sar, ConleySpatialHAC(xy[linked, :], Float64(bw)))))
    @printf("%-10d %14.4f %14.4f %10.4f\n", bw, se[2], se[3], se[4])
end

**The investment share does not survive every bandwidth.** Its standard error
goes from 0.020 model-based to 0.038–0.042 under Conley, about twice as wide.
At 300 km its z is 2.07. At 500 km it is 1.93 and at 800 km 1.90, below the 5%
line. The one stable coefficient from section 3 is significant only if
dependence in the errors dies out within about 300 km. Over a sample that spans
3,900 km, that is a claim, not a default.

The standard error of the initial-income coefficient barely moves. ρ̂'s
standard error *halves*, which is the warning sign seen in notebook `11`: a
sandwich around the spatial parameter of a model already fitted for spatial
dependence is not a better estimate of its precision.

## What this notebook is for

- **Check whether the "spatial" dependence is national.** Here country
  dummies took 60% of the variance in growth and turned ρ̂ from +0.575 to
  −0.225. A spatial lag over regions in several countries mixes the two.
- **A pooled convergence slope can be an average of opposite effects.** Across
  countries the poor caught up. Within them the rich pulled ahead, most
  strongly in the new member states.
- **When the LM battery and the likelihood disagree, say so.** Robust LM
  preferred the lag; AIC preferred SEM by 3 points; Vuong could not separate
  them. The data did not choose.
- **Look at the top of the table before you fit.** Ireland's GDP is an
  accounting artefact, and three adjacent outliers moved ρ̂ by 0.1.
- **Sweep the Conley bandwidth.** The investment coefficient is significant at
  300 km and not at 500 km.

## Exercises

1. Fit SDM with the country dummies. Do the neighbours' regressors still earn
   their place once the national level is absorbed, or was the SDM's AIC win
   in section 3 also national?
2. Replace the 2004 split with a rule: regions whose 2012 GDP per head was
   below 75% of the EU average, the threshold for the EU's cohesion funds. Does
   the Chow test still reject? Which split would you defend, and on what
   grounds other than the result?
3. Repeat section 4 with k-nearest-neighbour weights. With k = 6, many of a
   border region's neighbours are in another country. Does ρ̂ still turn
   negative with country dummies?
4. The panel runs from 1999. Build a two-way fixed-effects spatial panel of
   annual regional growth with `fit_spatial_panel` (see notebook `02`). The
   time effects absorb the euro crisis and COVID. What happens to ρ̂, and
   how does it compare with the cross-section?
5. The investment share is endogenous. Propose an instrument that a referee
   would accept, and say what it would have to be uncorrelated with.